## Part 0 — Reference data: NHTSA vPIC Make/Model lookup

Makes a handful of live API calls (needs internet, ~5–10s). Not required for the cleaning pipeline below.

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

VPIC_BASE = "https://vpic.nhtsa.dot.gov/api/vehicles"


In [2]:
df_cars = pd.read_csv(RAW_DIR / "used_cars.csv")

print("Shape:", df_cars.shape)
df_cars.head()


Shape: (99187, 11)


,Unnamed: 0,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Make
0,0,T-Roc,2019,25000,Automatic,13904,Diesel,145,49.6,2.0,VW
1,1,T-Roc,2019,26883,Automatic,4562,Diesel,145,49.6,2.0,VW
2,2,T-Roc,2019,20000,Manual,7414,Diesel,145,50.4,2.0,VW
3,3,T-Roc,2019,33492,Automatic,4825,Petrol,145,32.5,2.0,VW
4,4,T-Roc,2019,22900,Semi-Auto,6500,Petrol,150,39.8,1.5,VW


In [3]:
def get_all_makes() -> pd.DataFrame:
    """Fetch the full list of vehicle Makes recognized by NHTSA vPIC."""
    url = f"{VPIC_BASE}/GetAllMakes?format=json"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    results = resp.json()["Results"]
    return pd.DataFrame(results)

df_all_makes = get_all_makes()
print(df_all_makes.shape)
df_all_makes.head()


(12361, 2)


,Make_ID,Make_Name
0,12858,#1 ALPINE CUSTOMS
1,4877,"1/OFF KUSTOMS, LLC"
2,11257,"102 IRONWORKS, INC."
3,12255,12832429 CANADA INC.
4,13053,137 INDUSTRIES INC.


In [4]:
df_all_makes.to_csv(RAW_DIR / "vpic_all_makes.csv", index=False)
print("Saved:", RAW_DIR / "vpic_all_makes.csv")


Saved: ..\data\raw\vpic_all_makes.csv


In [5]:
# Raw `Make` values in this dataset use inconsistent casing/abbreviations
# (e.g. 'VW', 'vauxhall', 'merc', 'hyundi') rather than the official vPIC names.
# This map only covers the makes present in *this* dataset — extend it if you
# bring in more brands later, since anything missing here silently becomes NaN.
MAKE_MAP = {
    "VW": "Volkswagen", "vauxhall": "Vauxhall", "merc": "Mercedes-Benz",
    "hyundi": "Hyundai", "ford": "Ford", "toyota": "Toyota",
    "skoda": "Skoda", "BMW": "BMW", "Audi": "Audi",
}
df_cars["Make_clean"] = df_cars["Make"].map(MAKE_MAP)

unmapped = df_cars[df_cars["Make_clean"].isna()]["Make"].unique()
if len(unmapped):
    print("Warning — raw Make values with no entry in MAKE_MAP:", unmapped)

def get_models_for_make(make_name: str) -> pd.DataFrame:
    """Fetch all Models vPIC has on record for a given Make."""
    url = f"{VPIC_BASE}/GetModelsForMake/{make_name}?format=json"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    results = resp.json()["Results"]
    out = pd.DataFrame(results)
    out["queried_make"] = make_name
    return out

our_makes = sorted(df_cars["Make_clean"].dropna().unique())
print("Fetching models for:", our_makes)

model_frames = []
for make_name in our_makes:
    frame = get_models_for_make(make_name)
    print(f"  {make_name}: {len(frame)} models returned")
    model_frames.append(frame)
    time.sleep(0.5)  # be a polite API citizen

df_models = pd.concat(model_frames, ignore_index=True)
print("Total rows:", df_models.shape)


Fetching models for: ['Audi', 'BMW', 'Ford', 'Hyundai', 'Mercedes-Benz', 'Skoda', 'Toyota', 'Vauxhall', 'Volkswagen']
  Audi: 57 models returned
  BMW: 260 models returned
  Ford: 168 models returned
  Hyundai: 40 models returned
  Mercedes-Benz: 62 models returned
  Skoda: 0 models returned
  Toyota: 58 models returned
  Vauxhall: 0 models returned
  Volkswagen: 40 models returned
Total rows: (685, 5)


In [6]:
df_models.to_csv(RAW_DIR / "vpic_models_by_make.csv", index=False)
print("Saved:", RAW_DIR / "vpic_models_by_make.csv")


Saved: ..\data\raw\vpic_models_by_make.csv


## Part 1 — Load & initial inspection

Fresh read into `df` — this is the dataframe the rest of the notebook builds on.

In [7]:
df = pd.read_csv(RAW_DIR / "used_cars.csv")
df.head(10)


,Unnamed: 0,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Make
0,0,T-Roc,2019,25000,Automatic,13904,Diesel,145,49.6,2.0,VW
1,1,T-Roc,2019,26883,Automatic,4562,Diesel,145,49.6,2.0,VW
2,2,T-Roc,2019,20000,Manual,7414,Diesel,145,50.4,2.0,VW
3,3,T-Roc,2019,33492,Automatic,4825,Petrol,145,32.5,2.0,VW
4,4,T-Roc,2019,22900,Semi-Auto,6500,Petrol,150,39.8,1.5,VW
5,5,T-Roc,2020,31895,Manual,10,Petrol,145,42.2,1.5,VW
6,6,T-Roc,2020,27895,Manual,10,Petrol,145,42.2,1.5,VW
7,7,T-Roc,2020,39495,Semi-Auto,10,Petrol,145,32.5,2.0,VW
8,8,T-Roc,2019,21995,Manual,10,Petrol,145,44.1,1.0,VW
9,9,T-Roc,2019,23285,Manual,10,Petrol,145,42.2,1.5,VW


In [8]:
# Apply the same MAKE_MAP defined in Part 0 to standardize brand names in the
# actual pipeline dataframe -- Part 0 only cleaned the throwaway `df_cars` copy,
# so without this, 'VW', 'vauxhall', 'merc' etc. would flow straight into the saved file.
df['Make'] = df['Make'].map(MAKE_MAP)
df['Make'].unique()


array(['Volkswagen', 'Vauxhall', 'Toyota', 'Skoda', 'Mercedes-Benz',
       'Hyundai', 'Ford', 'BMW', 'Audi'], dtype=object)

**Insight:** check the list above -- you should see standardized names like 'Volkswagen', 'Vauxhall', 'Mercedes-Benz' rather than the raw abbreviations. If any value comes back as `NaN`, that means some raw `Make` wasn't covered by `MAKE_MAP` -- run `df['Make'].isna().sum()` to catch it before it silently drops out of any later groupby.


In [9]:
print("Shape:", df.shape)
df.info()


Shape: (99187, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99187 entries, 0 to 99186
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    99187 non-null  int64  
 1   model         99187 non-null  object 
 2   year          99187 non-null  int64  
 3   price         99187 non-null  int64  
 4   transmission  99187 non-null  object 
 5   mileage       99187 non-null  int64  
 6   fuelType      99187 non-null  object 
 7   tax           99187 non-null  int64  
 8   mpg           99187 non-null  float64
 9   engineSize    99187 non-null  float64
 10  Make          99187 non-null  object 
dtypes: float64(2), int64(5), object(4)
memory usage: 8.3+ MB


In [10]:
df.isnull().sum() / len(df) * 100


Unnamed: 0      0.0
model           0.0
year            0.0
price           0.0
transmission    0.0
mileage         0.0
fuelType        0.0
tax             0.0
mpg             0.0
engineSize      0.0
Make            0.0
dtype: float64

**Insight:** zero missing values across every column — this is clean source data. The only nulls we'll create ourselves are the deliberate `engineSize == 0 → NaN` conversion in Part 2, which then gets imputed.

In [11]:
df.duplicated().sum()


np.int64(0)

**Insight — this result is misleading.** `Unnamed: 0` is a per-row index column, so every
row is technically "unique" as long as it's included — `duplicated()` returns 0 here for the
wrong reason. Dropping it below re-runs the check on actual content.

In [12]:
df = df.drop(columns=['Unnamed: 0'])
df.duplicated().sum()


np.int64(1475)

**Insight:** with the index column gone, 1,475 rows (≈1.5% of the dataset) turn out to be exact content duplicates — same model, year, price, mileage, everything. These get dropped below.

In [13]:
df = df.drop_duplicates()
df.shape


(97712, 10)

In [14]:
df['model'] = df['model'].str.strip()


## Part 2 — Fixing `engineSize == 0`

A real engine can't have zero displacement, so `0` here almost certainly means "not recorded"
rather than a genuine reading — except possibly for Electric vehicles, which don't have a
traditional displacement figure at all.

In [15]:
df[df['engineSize'] == 0]['fuelType'].value_counts()


fuelType
Petrol      158
Diesel       69
Hybrid       38
Electric      2
Other         1
Name: count, dtype: int64

In [16]:
df[df['engineSize'] == 0][
    ['model', 'year', 'price', 'fuelType', 'engineSize', 'Make']
].head(20)


,model,year,price,fuelType,engineSize,Make
649,T-Roc,2019,22000,Petrol,0.0,Volkswagen
664,T-Roc,2018,23000,Diesel,0.0,Volkswagen
4734,Golf,2015,11800,Diesel,0.0,Volkswagen
4761,Golf,2019,18000,Petrol,0.0,Volkswagen
4768,Golf,2017,12600,Diesel,0.0,Volkswagen
6347,Passat,2018,17000,Petrol,0.0,Volkswagen
6351,Passat,2017,16000,Diesel,0.0,Volkswagen
6354,Passat,2018,19500,Petrol,0.0,Volkswagen
6356,Passat,2019,18500,Petrol,0.0,Volkswagen
11559,Tiguan,2017,19200,Diesel,0.0,Volkswagen


In [17]:
df[df['engineSize'] > 0]['fuelType'].value_counts()


fuelType
Petrol      53824
Diesel      40350
Hybrid       3021
Other         245
Electric        4
Name: count, dtype: int64

**Insight:** 268 rows have `engineSize == 0`, and all but 2 of them are Petrol, Diesel, or
Hybrid — vehicle types that always have a real displacement figure, so these 266 are clearly
missing data, not genuine zeros. The 2 Electric rows are more ambiguous (EVs legitimately have
no combustion displacement), but they're too small a group to special-case here — they'll go
through the same Make+model median imputation as everything else below. If EV-specific handling
matters for your use case later, this is the spot to revisit.

The fix: treat `0` as missing, then impute using the median `engineSize` for that specific
Make+model combination (a Golf's typical engine size is a better estimate than the fleet-wide
median), falling back to the global median for any Make+model group with no other data to
borrow from.

In [18]:
df['engineSize'] = df['engineSize'].replace(0, np.nan)
df['engineSize'] = df.groupby(['Make', 'model'])['engineSize'].transform(
    lambda x: x.fillna(x.median())
)
df['engineSize'] = df['engineSize'].fillna(df['engineSize'].median())

df['engineSize'].isnull().sum()


np.int64(0)

In [19]:
# Sanity check: did any single Make+model group have *all* missing engineSize values?
# If so, its median is NaN and it silently fell through to the global-median fallback
# instead of getting a genuinely group-specific estimate.
group_all_missing = df.groupby(['Make', 'model'])['engineSize'].apply(lambda x: x.isna().all())
group_all_missing[group_all_missing].index.tolist()


[]

In [20]:
df[df['year'] > 2026][['model', 'Make', 'year', 'price']]


,model,Make,year,price
77499,Fiesta,Ford,2060,6495


In [21]:
df[df['year'] < 1990][['model', 'Make', 'year', 'price']]


,model,Make,year,price
25994,Zafira,Vauxhall,1970,10495
53866,M Class,Mercedes-Benz,1970,24999


In [22]:
df = df[(df['year'] >= 1990) & (df['year'] <= 2026)].copy()
df.shape


(97709, 10)

In [23]:
df.describe()


,year,price,mileage,tax,mpg,engineSize
count,97709.000000,97709.000000,97709.000000,97709.000000,97709.000000,97709.000000
mean,2017.067394,16773.572823,23219.101884,120.138831,55.206047,1.668844
std,2.107849,9868.593414,21060.893971,63.354366,16.181724,0.552544
min,1996.000000,450.000000,1.000000,0.000000,0.300000,0.600000
25%,2016.000000,9999.000000,7673.000000,125.000000,47.100000,1.200000
50%,2017.000000,14470.000000,17682.000000,145.000000,54.300000,1.600000
75%,2019.000000,20750.000000,32500.000000,145.000000,62.800000,2.000000
max,2020.000000,159999.000000,323000.000000,580.000000,470.800000,6.600000


## Part 4 — `mpg` cleaning

Both tails of this column need a look — very high values might be legitimate hybrids, very low or absurdly high values are more likely data errors.

In [24]:
print("High mpg (>150):", (df['mpg'] > 150).sum())
print("Low mpg (<15):", (df['mpg'] < 15).sum())
print("Total rows:", len(df))


High mpg (>150): 273
Low mpg (<15): 34
Total rows: 97709


In [25]:
df[df['mpg'] > 150][['model', 'Make', 'mpg', 'fuelType']].head()


,model,Make,mpg,fuelType
1006,Golf,Volkswagen,166.2,Hybrid
1478,Golf,Volkswagen,188.3,Hybrid
1558,Golf,Volkswagen,166.2,Hybrid
1653,Golf,Volkswagen,156.9,Hybrid
1768,Golf,Volkswagen,166.2,Hybrid


**Insight:** the high end (>150 mpg) is almost entirely Hybrid Golfs in the 150s–180s — plausible for a hybrid's combined-cycle rating, so these are left alone.

In [26]:
df[df['mpg'] < 15].groupby(['Make', 'model', 'fuelType'])['mpg'].agg(['count', 'mean'])


count  mean
Make          model    fuelType             
BMW           3 Series Hybrid        8   8.8
              X3       Hybrid        6   5.5
Hyundai       Ioniq    Hybrid        4   1.1
Mercedes-Benz A Class  Hybrid        3   1.1
              G Class  Diesel        1  11.0
Toyota        C-HR     Petrol        1   6.0
              Hilux    Diesel       10   2.8
Volkswagen    Golf SV  Petrol        1   0.3

In [27]:
# Hypothesis: did engineSize accidentally get copied into the mpg column for these rows?
low_mpg = df[df['mpg'] < 15]
match_rate = (low_mpg['mpg'] == low_mpg['engineSize']).mean() * 100
print(f"{match_rate:.1f}% of low-mpg rows have mpg == engineSize")


0.0% of low-mpg rows have mpg == engineSize


In [28]:
df = df[df['mpg'] >= 15]
print("Rows remaining:", len(df))


Rows remaining: 97675


In [29]:
df['mpg'].sort_values(ascending=False).head(15)


83358    470.8
84018    470.8
81366    470.8
85534    470.8
83240    470.8
78516    470.8
78518    470.8
81015    470.8
81949    470.8
83528    470.8
86573    470.8
88268    470.8
83483    470.8
79604    470.8
87662    470.8
Name: mpg, dtype: float64

In [30]:
(df['mpg'] > 100).sum()


np.int64(568)

In [31]:
df[df['mpg'] == 470.8][['model', 'fuelType', 'engineSize', 'price']].head(10)


,model,fuelType,engineSize,price
77770,i3,Other,0.6,17100
78359,i3,Hybrid,0.6,19998
78516,i3,Hybrid,0.6,19998
78518,i3,Hybrid,0.6,21898
79604,i3,Hybrid,0.6,19980
80090,i3,Hybrid,0.6,19490
81015,i3,Hybrid,0.6,16482
81366,i3,Hybrid,0.6,14285
81651,i3,Hybrid,0.6,18500
81838,i3,Hybrid,0.6,19495


In [32]:
implausible_high_mpg = (df['mpg'] > 300).sum()
print(f"Rows with mpg > 300 (implausible): {implausible_high_mpg}")

df = df[df['mpg'] <= 300]
print("Rows remaining:", len(df))


Rows with mpg > 300 (implausible): 43
Rows remaining: 97632


In [33]:
df.describe()


,year,price,mileage,tax,mpg,engineSize
count,97632.000000,97632.000000,97632.000000,97632.000000,97632.000000,97632.000000
mean,2017.066884,16766.213270,23222.425506,120.156352,55.040630,1.669201
std,2.107995,9861.812255,21062.476205,63.326270,13.603719,0.552223
min,1996.000000,450.000000,1.000000,0.000000,17.800000,1.000000
25%,2016.000000,9999.000000,7677.750000,125.000000,47.100000,1.200000
50%,2017.000000,14450.000000,17685.000000,145.000000,54.300000,1.600000
75%,2019.000000,20750.000000,32500.000000,145.000000,62.800000,2.000000
max,2020.000000,159999.000000,323000.000000,580.000000,256.800000,6.600000


**Note:** `df` is about to be reassigned to the `cars.csv` data below. Keep a copy of the fully-cleaned `used_cars` frame first — Part 6 merges the two cleaned datasets back together.

In [34]:
df_used_cars = df.copy()
print(df_used_cars.shape)

(97632, 10)


In [35]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/used_cars_clean.csv", index=False)

# 'cars' dataset cleaning 

In [36]:
df = pd.read_csv("../data/raw/cars.csv", low_memory=False)
print(df.shape)
df.head()

(762091, 20)


,manufacturer,model,year,mileage,engine,transmission,drivetrain,fuel_type,mpg,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,driver_reviews_num,price_drop,price
0,Acura,ILX Hybrid 1.5L,2013,92945.0,"1.5L I-4 i-VTEC variable valve control, engine...",Automatic,Front-wheel Drive,Gasoline,39-38,Black,Parchment,0.0,0.0,0.0,Iconic Coach,NaN,4.4,12.0,300.0,13988.0
1,Acura,ILX Hybrid 1.5L,2013,47645.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Gray,Ebony,1.0,1.0,1.0,Kars Today,NaN,4.4,12.0,NaN,17995.0
2,Acura,ILX Hybrid 1.5L,2013,53422.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Bellanova White Pearl,Ebony,0.0,1.0,1.0,Weiss Toyota of South County,4.3,4.4,12.0,500.0,17000.0
3,Acura,ILX Hybrid 1.5L,2013,117598.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Polished Metal Metallic,NaN,0.0,1.0,1.0,Apple Tree Acura,NaN,4.4,12.0,675.0,14958.0
4,Acura,ILX Hybrid 1.5L,2013,114865.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,NaN,Ebony,1.0,0.0,1.0,Herb Connolly Chevrolet,3.7,4.4,12.0,300.0,14498.0


In [37]:
df.shape

(762091, 20)

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 762091 entries, 0 to 762090
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   manufacturer         762091 non-null  object 
 1   model                762091 non-null  object 
 2   year                 762091 non-null  int64  
 3   mileage              761585 non-null  float64
 4   engine               747041 non-null  object 
 5   transmission         752187 non-null  object 
 6   drivetrain           740529 non-null  object 
 7   fuel_type            739164 non-null  object 
 8   mpg                  620020 non-null  object 
 9   exterior_color       753232 non-null  object 
 10  interior_color       705116 non-null  object 
 11  accidents_or_damage  737879 non-null  float64
 12  one_owner            730608 non-null  float64
 13  personal_use_only    737239 non-null  float64
 14  seller_name          753498 non-null  object 
 15  seller_rating    

In [39]:
df.isnull().sum() / len(df) * 100

manufacturer            0.000000
model                   0.000000
year                    0.000000
mileage                 0.066396
engine                  1.974830
transmission            1.299582
drivetrain              2.829321
fuel_type               3.008433
mpg                    18.642262
exterior_color          1.162460
interior_color          7.476141
accidents_or_damage     3.177048
one_owner               4.131134
personal_use_only       3.261028
seller_name             1.127556
seller_rating          28.077093
driver_rating           4.150685
driver_reviews_num      0.000000
price_drop             46.185954
price                   0.000000
dtype: float64

In [40]:
df.duplicated().sum()

np.int64(9145)

In [41]:
df.duplicated().value_counts()

False    752946
True       9145
Name: count, dtype: int64

In [42]:
df.describe()

,year,mileage,accidents_or_damage,one_owner,personal_use_only,seller_rating,driver_rating,driver_reviews_num,price_drop,price
count,762091.000000,7.615850e+05,737879.000000,730608.000000,737239.000000,548118.000000,730459.000000,762091.000000,410112.000000,7.620910e+05
mean,2017.791398,5.578169e+04,0.228616,0.561969,0.657212,4.158568,4.623523,89.550900,1007.467068,3.648898e+04
std,5.110532,4.355788e+04,0.419942,0.496145,0.474642,0.805741,0.276902,115.082266,1375.122208,1.984183e+06
min,1915.000000,0.000000e+00,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,100.000000,1.000000e+00
25%,2016.000000,2.328700e+04,0.000000,0.000000,0.000000,3.800000,4.500000,14.000000,380.000000,1.958300e+04
50%,2019.000000,4.559600e+04,0.000000,1.000000,1.000000,4.500000,4.700000,51.000000,642.000000,2.798900e+04
75%,2021.000000,7.836500e+04,0.000000,1.000000,1.000000,4.700000,4.800000,119.000000,1007.000000,3.948800e+04
max,2024.000000,1.119067e+06,1.000000,1.000000,1.000000,5.000000,5.000000,1025.000000,170995.000000,1.000000e+09


**Insight:** 9,145 exact duplicate rows (~1.2% of the dataset) — the same class of issue as `used_cars`. Drop them before anything else so later stats aren't skewed by repeated listings.

In [43]:
df = df.drop_duplicates()
print("Shape after dropping exact duplicates:", df.shape)

Shape after dropping exact duplicates: (752946, 20)


In [44]:
df['model'] = df['model'].str.strip()
df['manufacturer'] = df['manufacturer'].str.strip()

## Part 5b — Standardizing categorical fields



`drivetrain` and `fuel_type` both mix full names, abbreviations, and inconsistent casing — plus a handful of rows where an unrelated free-text value (an engine spec string, or the word `'Automatic'`) leaked into the column. Standardize both the same way `Make` was standardized for `used_cars` in Part 0.

In [45]:
print(df['drivetrain'].value_counts(dropna=False))

drivetrain
Front-wheel Drive                                              238528
All-wheel Drive                                                227752
Four-wheel Drive                                               155512
Rear-wheel Drive                                                96461
NaN                                                             20891
FWD                                                              6245
AWD                                                              3424
4WD                                                              1834
RWD                                                              1698
All-Wheel Drive                                                   115
Unknown                                                           102
Front-Wheel Drive                                                  97
Front-Wheel Drive with Limited-Slip Differential                   42
Four-Wheel Drive                                                   37
Four-Whee

In [46]:
def clean_drivetrain(val):
    if pd.isna(val):
        return np.nan
    v = str(val).strip()
    vl = v.lower()
    if vl.startswith('engine:'):
        # a handful of rows have an engine spec string leaked into this column
        return np.nan
    if vl in ('unknown', '2wd'):
        return np.nan
    if vl == '4x2':
        return 'Rear-Wheel Drive'
    if 'front' in vl or vl == 'fwd':
        return 'Front-Wheel Drive'
    if 'all' in vl or vl == 'awd':
        return 'All-Wheel Drive'
    if 'four' in vl or vl in ('4wd', '4x4'):
        return 'Four-Wheel Drive'
    if 'rear' in vl or vl == 'rwd':
        return 'Rear-Wheel Drive'
    return np.nan

df['drivetrain'] = df['drivetrain'].map(clean_drivetrain)
print(df['drivetrain'].value_counts(dropna=False))

drivetrain
Front-Wheel Drive    244929
All-Wheel Drive      231347
Four-Wheel Drive     157455
Rear-Wheel Drive      98212
NaN                   21003
Name: count, dtype: int64


In [47]:
print(df['fuel_type'].value_counts(dropna=False))

fuel_type
Gasoline                         637168
Hybrid                            28926
Diesel                            27681
NaN                               22153
E85 Flex Fuel                     18518
Electric                          16117
B                                  1410
Flexible Fuel                       468
Plug-In Hybrid                      118
Gasoline Fuel                        77
Gasoline/Mild Electric Hybrid        70
Regular Unleaded                     54
Premium Unleaded                     48
G                                    44
Compressed Natural Gas               29
Unspecified                          25
Diesel Fuel                           5
Hybrid Fuel                           5
Hydrogen Fuel Cell                    3
Gaseous                               3
Other                                 3
Flex Fuel Capability                  2
Gas/Electric Hybrid                   2
Gas                                   2
PHEV                          

In [48]:
def clean_fuel_type(val):
    if pd.isna(val):
        return np.nan
    v = str(val).strip()
    vl = v.lower()
    if vl in ('unspecified', 'unknown', 'other', 'automatic', 'gaseous'):
        # 'automatic' is a transmission value that leaked into this column
        return np.nan
    if 'hydrogen' in vl:
        return 'Hydrogen'
    if 'plug-in' in vl or vl == 'phev':
        return 'Plug-In Hybrid'
    if 'hybrid' in vl:
        return 'Hybrid'
    if 'electric' in vl:
        return 'Electric'
    if 'flex' in vl or vl == 'e85 fl' or 'bi-fuel' in vl:
        return 'Flex Fuel'
    if 'diesel' in vl or vl == 'b':
        return 'Diesel'
    if 'natural gas' in vl:
        return 'Natural Gas'
    if 'gas' in vl or vl == 'g' or 'unleaded' in vl or 'premium' in vl:
        return 'Gasoline'
    return np.nan

df['fuel_type'] = df['fuel_type'].map(clean_fuel_type)
print(df['fuel_type'].value_counts(dropna=False))

fuel_type
Gasoline          637395
Diesel             29098
Hybrid             29003
NaN                22186
Flex Fuel          18991
Electric           16119
Plug-In Hybrid       121
Natural Gas           30
Hydrogen               3
Name: count, dtype: int64


In [49]:
mpg_str = df['mpg'].dropna().astype(str)
print("Rows shaped like 'city-highway':", mpg_str.str.contains('-').sum())
print("Rows with a single combined number:", (~mpg_str.str.contains('-')).sum())
print("Rows literally '0-0':", (mpg_str == '0-0').sum())

Rows shaped like 'city-highway': 610236
Rows with a single combined number: 2820
Rows literally '0-0': 7203


In [50]:
def parse_mpg(val):
    if pd.isna(val):
        return (np.nan, np.nan)
    s = str(val).strip()
    if '-' in s:
        parts = s.split('-')
        try:
            city, hwy = float(parts[0]), float(parts[1])
        except ValueError:
            return (np.nan, np.nan)
    else:
        try:
            city = hwy = float(s)
        except ValueError:
            return (np.nan, np.nan)
    city = np.nan if city == 0 else city
    hwy = np.nan if hwy == 0 else hwy
    return (city, hwy)

parsed = df['mpg'].apply(parse_mpg)
df['city_mpg'] = parsed.apply(lambda t: t[0])
df['highway_mpg'] = parsed.apply(lambda t: t[1])
df = df.drop(columns=['mpg'])
print("Missing city_mpg:", df['city_mpg'].isna().sum())
print("Missing highway_mpg:", df['highway_mpg'].isna().sum())

Missing city_mpg: 147705
Missing highway_mpg: 150125


In [51]:
for col in ['city_mpg', 'highway_mpg']:
    df[col] = df.groupby(['manufacturer', 'model'])[col].transform(lambda x: x.fillna(x.median()))
    df[col] = df[col].fillna(df[col].median())
print("Missing city_mpg after impute:", df['city_mpg'].isna().sum())
print("Missing highway_mpg after impute:", df['highway_mpg'].isna().sum())

Missing city_mpg after impute: 0
Missing highway_mpg after impute: 0


**Insight:** `mileage` has 3 rows sitting at exactly `999999` — an all-nines sentinel for "unknown," not a real odometer reading. `mileage == 0` (1,480 rows) is left alone — plausible for new or dealer-demo listings on a marketplace like this, unlike `engineSize == 0` which can't be a real reading for a combustion engine. Treat the sentinel as missing and impute the same way as `mpg`/`engineSize`.

In [52]:
print("mileage == 999999 (sentinel):", (df['mileage'] == 999999).sum())
print("mileage == 0 (plausible new/demo cars):", (df['mileage'] == 0).sum())

mileage == 999999 (sentinel): 3
mileage == 0 (plausible new/demo cars): 1480


In [53]:
df['mileage'] = df['mileage'].replace(999999, np.nan)
df['mileage'] = df.groupby(['manufacturer', 'model'])['mileage'].transform(lambda x: x.fillna(x.median()))
df['mileage'] = df['mileage'].fillna(df['mileage'].median())
print("Missing mileage after impute:", df['mileage'].isna().sum())

Missing mileage after impute: 0


**Insight:** `price_drop` is null for ~46% of rows, but every non-null value is >= $100 and there isn't a single recorded $0 drop. That means null encodes "no price drop happened," not "unknown" — so it should be filled with 0 rather than imputed or left missing.

In [54]:
print("price_drop non-null count:", df['price_drop'].notna().sum())
print("price_drop == 0 count (should be 0):", (df['price_drop'] == 0).sum())
print("price_drop min among non-null:", df['price_drop'].min())

price_drop non-null count: 406236
price_drop == 0 count (should be 0): 0
price_drop min among non-null: 100.0


In [55]:
df['price_drop'] = df['price_drop'].fillna(0)
print("Missing price_drop after fill:", df['price_drop'].isna().sum())

Missing price_drop after fill: 0


In [56]:
# tidy up the 0/1/NaN flag columns into a proper nullable boolean dtype
for col in ['accidents_or_damage', 'one_owner', 'personal_use_only']:
    df[col] = df[col].astype('boolean')
print(df[['accidents_or_damage', 'one_owner', 'personal_use_only']].dtypes)

accidents_or_damage    boolean
one_owner              boolean
personal_use_only      boolean
dtype: object


**Insight:** `price` has a legitimate long tail of real exotics — a Porsche Carrera GT and 918 Spyder list up to ~$2.25M, which checks out against real-world market values — but a handful of rows are obvious data-entry garbage ($1,000,000,000 listings, an $8.9M police Interceptor, a $3.49M cargo van), and 14 rows sit under $500 for ordinary low-mileage cars (a 2018 Acura for $1, a 2021 Nissan Versa for $259), which isn't a real price for a working car. Drop both tails at thresholds set just outside the real exotics.

In [57]:
print("Rows priced under $500:", (df['price'] < 500).sum())
print(df[df['price'] < 500][['manufacturer', 'model', 'year', 'mileage', 'price']].sort_values('price').head(5))
print()
print("Rows priced over $2.5M:", (df['price'] > 2_500_000).sum())
print(df[df['price'] > 2_500_000][['manufacturer', 'model', 'year', 'mileage', 'price']].sort_values('price', ascending=False))

Rows priced under $500: 14
       manufacturer                model  year   mileage  price
5658          Acura        TLX V6 A-Spec  2018   49603.0    1.0
13850          Audi      A4 2.0T Premium  2014   98881.0    1.0
660696       Subaru  Legacy 2.5i Limited  2019   31570.0    1.0
735177   Volkswagen                Jetta  2007  133964.0    1.0
584638       Nissan             Versa SV  2021   45288.0  259.0

Rows priced over $2.5M: 5
       manufacturer                            model  year   mileage  \
108142    Chevrolet                        Cobalt LT  2009   85185.0   
188113        Dodge                  Durango Citadel  2018  113207.0   
224571         Ford  Utility Police Interceptor Base  2016   96847.5   
84358      Cadillac      DeVille 77 HOURS ON ENGINES  1963      76.0   
636956          RAM         ProMaster 3500 High Roof  2017   83000.0   

               price  
108142  1.000000e+09  
188113  1.000000e+09  
224571  8.888889e+06  
84358   4.999999e+06  
636956  3.4900

In [58]:
before = len(df)
df = df[(df['price'] >= 500) & (df['price'] <= 2_500_000)]
print("Dropped implausible price rows:", before - len(df))
print("Shape:", df.shape)

Dropped implausible price rows: 19
Shape: (752927, 21)


In [59]:
# a few of the imputation steps above can coincidentally turn near-duplicate rows into exact ones
dupe_ct = df.duplicated().sum()
print("Duplicate rows created by imputation:", dupe_ct)
df = df.drop_duplicates()
print("Final shape:", df.shape)

Duplicate rows created by imputation: 27
Final shape: (752900, 21)


In [60]:
print(df.isnull().sum() / len(df) * 100)

manufacturer            0.000000
model                   0.000000
year                    0.000000
mileage                 0.000000
engine                  1.919777
transmission            1.258866
drivetrain              2.789348
fuel_type               2.946474
exterior_color          1.158454
interior_color          7.419445
accidents_or_damage     3.141320
one_owner               4.085802
personal_use_only       3.225794
seller_name             1.128171
seller_rating          28.089786
driver_rating           4.154204
driver_reviews_num      0.000000
price_drop              0.000000
price                   0.000000
city_mpg                0.000000
highway_mpg             0.000000
dtype: float64


**Insight — updated:** the three accident/ownership flags are genuine missing data (the listing simply didn't report it) and stay `NaN`. Everything else below is filled rather than left open: `seller_rating`/`driver_rating` with the median for that exact manufacturer+model (global median as fallback, plus a `_was_missing` flag per column so imputed values stay identifiable), and the remaining text columns (`engine`, `transmission`, `drivetrain`, `fuel_type`, both color columns, `seller_name`) with the explicit string `"Unknown"`.

In [61]:
numeric_impute_cols = ['seller_rating', 'driver_rating']
for col in numeric_impute_cols:
    df[col + '_was_missing'] = df[col].isna()
    df[col] = df.groupby(['manufacturer', 'model'])[col].transform(lambda x: x.fillna(x.median()))
    df[col] = df[col].fillna(df[col].median())

categorical_fill_cols = ['engine', 'transmission', 'drivetrain', 'fuel_type',
                          'exterior_color', 'interior_color', 'seller_name']
for col in categorical_fill_cols:
    df[col] = df[col].fillna('Unknown')

print(df.isnull().sum() / len(df) * 100)

manufacturer                 0.000000
model                        0.000000
year                         0.000000
mileage                      0.000000
engine                       0.000000
transmission                 0.000000
drivetrain                   0.000000
fuel_type                    0.000000
exterior_color               0.000000
interior_color               0.000000
accidents_or_damage          3.141320
one_owner                    4.085802
personal_use_only            3.225794
seller_name                  0.000000
seller_rating                0.000000
driver_rating                0.000000
driver_reviews_num           0.000000
price_drop                   0.000000
price                        0.000000
city_mpg                     0.000000
highway_mpg                  0.000000
seller_rating_was_missing    0.000000
driver_rating_was_missing    0.000000
dtype: float64


In [62]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/cars_clean.csv", index=False)

print(" ../data/processed/cars_clean.csv", df.shape)


 ../data/processed/cars_clean.csv (752900, 23)
